# Stage3 FP32 controls
Training-only, scratch regularized / scratch unregularized / trained unregularized. Whole runner budget 3300 seconds.

In [ ]:
import time
RUN_STARTED = time.monotonic()
from pathlib import Path, PurePosixPath
import hashlib, json, shutil, tempfile, zipfile

INPUT = Path('/kaggle/input')
WORK_PARENT = Path('/kaggle/working')
# Set SOURCE explicitly only when several matching bundles are attached.
SOURCE = None
if SOURCE is None:
    archives = list(INPUT.rglob('stage3-training-v1.zip'))
    directories = [p.parent for p in INPUT.rglob('bundle.json')
                   if (p.parent/'src/stage3_pipeline.py').is_file()
                   and (p.parent/'dataset/split_manifest.csv').is_file()]
    candidates = archives + directories
    if len(candidates) != 1:
        raise RuntimeError(f'Expected one Stage3 bundle; set SOURCE to one of: {candidates}')
    SOURCE = candidates[0]
SOURCE = Path(SOURCE)
WORK = Path(tempfile.mkdtemp(prefix='stage3-gpu-', dir=WORK_PARENT))
if SOURCE.is_file():
    with zipfile.ZipFile(SOURCE) as z:
        for member in z.infolist():
            rel = PurePosixPath(member.filename)
            if rel.is_absolute() or '..' in rel.parts or '\\' in member.filename:
                raise ValueError('Unsafe ZIP member')
            if not (WORK/member.filename).resolve().is_relative_to(WORK.resolve()):
                raise ValueError('ZIP path escapes workspace')
        z.extractall(WORK)
else:
    shutil.copytree(SOURCE, WORK, dirs_exist_ok=True)
BUNDLE = json.loads((WORK/'bundle.json').read_text())
if BUNDLE.get('kind') != 'STAGE3_FULL_V1':
    raise ValueError('Not a Stage3 smoke bundle')
for relative, expected in BUNDLE['file_sha256'].items():
    path = (WORK/relative).resolve()
    if not path.is_relative_to(WORK.resolve()):
        raise ValueError('Manifest path escapes workspace')
    if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
        raise ValueError(f'Bundle checksum mismatch: {relative}')
print('Verified bundle:', SOURCE)
print('Working directory:', WORK)

assets=list(INPUT.rglob('diagnostic-assets.json'))
if len(assets)!=1:raise RuntimeError('Expected diagnostic assets')
ASSET=assets[0].parent
for name,expected in json.loads(assets[0].read_text())['files'].items():
    if hashlib.sha256((ASSET/name).read_bytes()).hexdigest()!=expected:raise ValueError('Asset hash mismatch')
    shutil.copyfile(ASSET/name,WORK/name if name.endswith('.pt') else WORK/'src'/name)


In [ ]:
import subprocess, sys, time, os
VENV = Path(tempfile.mkdtemp(prefix='stage3-pinned-', dir='/tmp'))
subprocess.run([sys.executable, '-m', 'venv', '--without-pip', str(VENV)], check=True)
PYTHON = str(VENV/'bin/python')
started = time.monotonic()
with (WORK/'install.log').open('w') as log:
    installed = subprocess.run([sys.executable, '-m', 'pip', '--python', PYTHON, 'install', '--no-cache-dir', '-r', str(WORK/'requirements.txt')], stdout=log, stderr=subprocess.STDOUT, timeout=600)
INSTALL = {'exit_code': installed.returncode, 'seconds': time.monotonic()-started,
           'requirements_sha256': hashlib.sha256((WORK/'requirements.txt').read_bytes()).hexdigest()}
(WORK/'install.json').write_text(json.dumps(INSTALL, indent=2))
if installed.returncode:
    print((WORK/'install.log').read_text()[-12000:])
    raise RuntimeError('Pinned requirements installation failed')
print('Installation:', INSTALL)
remaining = min(2500, int(3200-(time.monotonic()-RUN_STARTED)))
if remaining < 120: raise RuntimeError('Insufficient remaining experiment budget')
with (WORK/'diagnostic.log').open('w') as log:
    run = subprocess.run([PYTHON,'-u','src/diagnose_stage3_precision.py','--dataset-dir','dataset','--checkpoint','diagnostic-best.pt','--output-dir','diagnostic-run','--steps','240','--max-seconds',str(remaining-60)],cwd=WORK,stdout=log,stderr=subprocess.STDOUT,timeout=remaining)
print((WORK/'diagnostic.log').read_text()[-14000:])
with zipfile.ZipFile(WORK/'stage3-diagnostic-result.zip','w',zipfile.ZIP_DEFLATED) as z:
    for name in ['diagnostic-run/diagnostic.json','diagnostic.log','install.json']:
        if (WORK/name).exists(): z.write(WORK/name,name)
if run.returncode: raise RuntimeError('FP32 diagnostic failed; inspect log')
print('Runner seconds:', time.monotonic()-RUN_STARTED)
